# Text Classification

In [1]:
import re
import string
import nltk
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [4]:
def text_cleaning_pipeline(text, rule="lemmatize"):
    """
    Clean the input text using a set of preprocessing steps:
    - Lowercase conversion
    - URL and emoji removal
    - Punctuation removal
    - Stopword removal
    - Lemmatization or stemming
    """

    # 1. Lowercase the text
    data = text.lower()

    # 2. Remove URLs
    data = re.sub(r"http\S+|www\S+|https\S+", '', data, flags=re.MULTILINE)

    # 3. Remove emojis and non-ASCII characters
    data = re.sub(r'[^\x00-\x7F]+', '', data)

    # 4. Remove punctuation
    data = data.translate(str.maketrans('', '', string.punctuation))

    # 5. Tokenize
    tokens = data.split()

    # 6. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # 7. Lemmatize or stem
    if rule == "lemmatize":
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    elif rule == "stem":
        tokens = [stemmer.stem(word) for word in tokens]
    else:
        print("Invalid rule. Pick between 'lemmatize' or 'stem'.")

    return " ".join(tokens)


In [5]:
data = pd.read_csv("/content/drive/MyDrive/ai Workshop Dataset/trum_tweet_sentiment_analysis.csv", encoding="ISO-8859-1")
data.head()

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [6]:
print(data.columns)

Index(['text', 'Sentiment'], dtype='object')


In [7]:
data_cleaning = data["text"].dropna()

In [8]:
cleaned_tokens = data["text"].apply(lambda dataset: text_cleaning_pipeline(dataset))


In [9]:
# Split the data into features (X) and target (y)
X = data['text']
y = data['text']

In [10]:
# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=1000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
# Logistic Regression model
model = LogisticRegression(max_iter=150)
model.fit(X_train_vec, y_train)


In [ ]:
y_pred = model.predict(X_test_vec)

In [ ]:
# Print classification report
print(classification_report(y_test, y_pred))